# 14 FORECAST: daily fault-count forecasting

SARIMA, an LSTM, and naive and seasonal naive baselines over a common 30-day blind horizon (22 June to 21 July 2025).

* **SARIMA.** No differencing (d = 0, D = 0) and a constant, with orders chosen by AIC; the comparison with a seasonally differenced alternative is in `15_SARIMA`.
* **LSTM.** Forecasts recursively from its own outputs, with the scaler fitted on the training partition only. Its prediction interval, widened with the square root of the forecast step, is approximate. Needs the Python 3.11 environment with TensorFlow.
* **Accuracy.** MAE, RMSE and MASE; the series contains zeros, so percentage errors are undefined.
* **Out-of-service days.** Runs of `gap_min_days` or more consecutive zero-count days are detected automatically, printed, and masked as missing rather than counted as fault-free days. Masked days are excluded from the stationarity tests, the dispersion fit, the ACF, the MASE scale and the LSTM training sequences; SARIMAX handles them through the Kalman filter.

Input `build/fault_episodes_v4.csv`; outputs in `OUTPUT/v4/`. Requires `statsmodels`, `tensorflow-cpu` and `tqdm`.

Paper: Sections 2.7.4 and 3.3; Table 11; Figures 7 and 8.

In [ ]:
# =====================================================================
# Fault count forecasting
# =====================================================================
import os, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
from tqdm.auto import tqdm
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.discrete.discrete_model import NegativeBinomial
from sklearn.preprocessing import MinMaxScaler

CFG = dict(
    episodes="build/fault_episodes_v4.csv",
    outdir="OUTPUT/v4",
    horizon=30,
    m=7,
    lookback=14,
    seed=42,
    gap_min_days=7,
)

In [ ]:
# ---------------------------------------------------------------------
# 1. Daily counts

In [ ]:
# ---------------------------------------------------------------------
def daily_counts(cfg):
    ep = pd.read_csv(cfg["episodes"], parse_dates=["onset"])
    s = (ep.set_index("onset").resample("D").size()
           .asfreq("D").fillna(0).astype(float))
    s.name = "faults"

    zero = s.eq(0)
    block = (zero != zero.shift()).cumsum()
    masked = []
    for _, idx in s.groupby(block).groups.items():
        idx = pd.DatetimeIndex(idx)
        if zero.loc[idx].all() and len(idx) >= cfg["gap_min_days"]:
            s.loc[idx] = np.nan
            masked.append((idx.min().date(), idx.max().date(), len(idx)))

    if masked:
        print(f"GAPS MASKED (zero-runs of >= {cfg['gap_min_days']} days, "
              f"unit out of service, not observed zeros):")
        for a, b, n in masked:
            print(f"    {a} to {b}   ({n} days)")
    else:
        print(f"No zero-runs of >= {cfg['gap_min_days']} days found.")
    return s


In [ ]:
# ---------------------------------------------------------------------
# 2. Stationarity and distribution

In [ ]:
# ---------------------------------------------------------------------
def diagnose(train, cfg):
    obs = train.dropna()
    print(f"\n  observed {len(obs)} of {len(train)} training days "
          f"({train.isna().sum()} masked)\n")
    print("STATIONARITY   [observed days only]")
    for name, d in [("level", obs), ("1st difference", obs.diff().dropna())]:
        a = adfuller(d, autolag="AIC")
        k = kpss(d, regression="c", nlags="auto")
        print(f"  {name:16s} ADF p={a[1]:.4f}   KPSS p={k[1]:.4f}"
              f"   -> {'stationary' if a[1] < .05 and k[1] > .05 else 'see both'}")

    print("\nDISPERSION   [observed days only]")
    mu, var = obs.mean(), obs.var()
    print(f"  mean {mu:.2f}   variance {var:.2f}   ratio {var/mu:.2f}")
    try:
        y = obs.values
        nb = NegativeBinomial(y, np.ones((len(y), 1))).fit(disp=0)
        print(f"  negative binomial alpha = {nb.params[-1]:.3f} "
              f"(alpha > 0 confirms overdispersion; Poisson would give 0)")
    except Exception as e:
        print(f"  negative binomial fit failed: {e}")

    r = acf(train, nlags=cfg["m"] * 3, fft=False, missing="conservative")
    print(f"\n  ACF at lag {cfg['m']}: {r[cfg['m']]:.3f}"
          f"   lag {cfg['m']*2}: {r[cfg['m']*2]:.3f}   [gap-aware]")

In [ ]:
# ---------------------------------------------------------------------
# 3. Baselines

In [ ]:
# ---------------------------------------------------------------------
def naive(train, h):
    return np.repeat(train.dropna().iloc[-1], h)


def seasonal_naive(train, h, m):
    last = train.ffill().iloc[-m:].values
    return np.array([last[i % m] for i in range(h)])

In [ ]:
# ---------------------------------------------------------------------
# 4. SARIMA

In [ ]:
def fit_sarima(train, h, cfg, grid=None):
    """Specification: d = 0 and D = 0 with a constant; orders and the
    undifferenced seasonal form by AIC, which is comparable within this family."""
    grid = grid or [(p, 0, q) for p in (0, 1, 2) for q in (0, 1, 2)]
    seas = [(0, 0, 0, 0), (1, 0, 0, cfg["m"]), (0, 0, 1, cfg["m"]), (1, 0, 1, cfg["m"])]
    best = None
    for o in tqdm(grid, desc="SARIMA grid", leave=False):
        for s in seas:
            try:
                r = SARIMAX(train, order=o, seasonal_order=s, trend="c",
                            enforce_stationarity=False,
                            enforce_invertibility=False).fit(disp=False)
                if best is None or r.aic < best[0]:
                    best = (r.aic, o, s, r)
            except Exception:
                continue
    aic, o, s, res = best
    print(f"  selected SARIMA{o}{s} with constant  AIC={aic:.1f}")
    fc = res.get_forecast(steps=h)
    return fc.predicted_mean.values, fc.conf_int(alpha=0.05).values, (o, s)

In [ ]:
# ---------------------------------------------------------------------
# 5. LSTM -- genuine recursive multi-step forecast

In [ ]:
# ---------------------------------------------------------------------
def fit_lstm(train, h, cfg):
    import tensorflow as tf
    tf.random.set_seed(cfg["seed"]); np.random.seed(cfg["seed"])
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Input

    L = cfg["lookback"]
    sc = MinMaxScaler().fit(train.dropna().values.reshape(-1, 1))
    z = sc.transform(train.values.reshape(-1, 1)).ravel()

    keep = [i for i in range(len(z) - L) if not np.isnan(z[i:i + L + 1]).any()]
    print(f"  {len(keep)} training sequences "
          f"({len(z) - L - len(keep)} dropped for spanning a masked gap)")
    X = np.array([z[i:i + L] for i in keep])[..., None]
    y = np.array([z[i + L] for i in keep])
    cut = int(len(X) * 0.85)
    Xtr, ytr, Xva, yva = X[:cut], y[:cut], X[cut:], y[cut:]

    model = Sequential([Input((L, 1)), LSTM(50), Dense(1)])
    model.compile(optimizer="adam", loss="mse")
    model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=200, batch_size=16,
              verbose=0, callbacks=[tf.keras.callbacks.EarlyStopping(
                  patience=20, restore_best_weights=True)])

    tail = train.ffill().bfill().values[-L:].reshape(-1, 1)
    window = list(sc.transform(tail).ravel())
    preds = []
    for _ in range(h):
        p = float(model.predict(np.array(window[-L:])[None, :, None], verbose=0)[0, 0])
        preds.append(p)
        window.append(p)
    pred = sc.inverse_transform(np.array(preds).reshape(-1, 1)).ravel()

    va_pred = model.predict(Xva, verbose=0).ravel()
    resid = sc.inverse_transform(yva.reshape(-1, 1)).ravel() - \
            sc.inverse_transform(va_pred.reshape(-1, 1)).ravel()
    lo, hi = np.quantile(resid, [0.025, 0.975])
    grow = np.sqrt(np.arange(1, h + 1))
    ci = np.column_stack([pred + lo * grow, pred + hi * grow])
    return pred, ci

In [ ]:
# ---------------------------------------------------------------------
# 6. Metrics

In [ ]:
# ---------------------------------------------------------------------
def score(y, pred, train, m):
    y, pred = np.asarray(y, float), np.asarray(pred, float)
    mae = np.mean(np.abs(y - pred))
    rmse = np.sqrt(np.mean((y - pred) ** 2))
    d = train.values[m:] - train.values[:-m]
    scale = np.nanmean(np.abs(d))
    return dict(MAE=mae, RMSE=rmse, MASE=mae / scale if scale else np.nan)

In [ ]:
# ---------------------------------------------------------------------
# 7. Run

In [ ]:
# ---------------------------------------------------------------------
def run(cfg, use_prophet=False):
    os.makedirs(cfg["outdir"], exist_ok=True)
    s = daily_counts(cfg)
    h = cfg["horizon"]
    train, test = s.iloc[:-h], s.iloc[-h:]
    print(f"\n{len(s)} days: {s.index.min().date()} to {s.index.max().date()}")
    print(f"train {len(train)}  test {len(test)}"
          f"   test window {test.index.min().date()} to {test.index.max().date()}")
    if test.isna().any():
        print("  WARNING: the test window contains masked days.")
    diagnose(train, cfg)

    print("\nFORECASTS")
    out, ci = {}, {}
    out["Naive"] = naive(train, h)
    out[f"Seasonal naive (m={cfg['m']})"] = seasonal_naive(train, h, cfg["m"])
    out["SARIMA"], ci["SARIMA"], order = fit_sarima(train, h, cfg)
    out["LSTM"], ci["LSTM"] = fit_lstm(train, h, cfg)

    if use_prophet:
        try:
            from prophet import Prophet
            d = train.reset_index(); d.columns = ["ds", "y"]
            mp = Prophet(weekly_seasonality=True, daily_seasonality=False).fit(d)
            fut = mp.make_future_dataframe(periods=h)
            out["Prophet"] = mp.predict(fut)["yhat"].values[-h:]
        except Exception as e:
            print(f"  Prophet skipped: {e}")

    res = pd.DataFrame({k: score(test, v, train, cfg["m"])
                        for k, v in out.items()}).T.round(3)
    res = res.sort_values("MASE")
    print("\n" + res.to_string())
    print("\nMASE < 1 beats the seasonal naive baseline; MASE >= 1 does not.")
    res.to_csv(f"{cfg['outdir']}/forecast_metrics.csv")
    pd.DataFrame(out, index=test.index).assign(actual=test.values).to_csv(
        f"{cfg['outdir']}/forecasts.csv")
    return s, train, test, out, ci, res

## Run

In [ ]:
s, train, test, out, ci, res = run(CFG, use_prophet=False)

## Forecast plot (Figure 7)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 9.5, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.7, "figure.dpi": 110,
})

TAIL = 120
STYLE = {"SARIMA":  dict(c="#1f6fb4", ls="-",  lw=1.7, z=5),
         "LSTM":    dict(c="#c0392b", ls="-",  lw=1.7, z=5),
         f"Seasonal naive (m={CFG['m']})": dict(c="#e08a1e", ls="--", lw=1.2, z=4),
         "Naive":   dict(c="#7f8c8d", ls=":",  lw=1.2, z=4)}

hist = s.iloc[-TAIL:]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw=dict(width_ratios=[2.1, 1], wspace=0.12))

# ---- left: context -------------------------------------------------
na = s.isna()
blk = (na != na.shift()).cumsum()
shown = False
for _, g in s.groupby(blk):
    if na.loc[g.index].all() and g.index[-1] >= hist.index[0]:
        axL.axvspan(max(g.index[0], hist.index[0]), g.index[-1], color="0.90", lw=0,
                    zorder=0, label=None if shown else "Out of service (annual maintenance)")
        shown = True
axL.axvspan(test.index[0], test.index[-1], color="#eef3fa", lw=0, zorder=0)
axL.plot(hist.index, hist.values, color="0.25", lw=1.0, label="Observed", zorder=6)
axL.axvline(train.index[-1], color="0.2", ls=":", lw=0.8, zorder=7)
axL.annotate("forecast\norigin", xy=(train.index[-1], 0.98),
             xycoords=("data", "axes fraction"), xytext=(-4, 0),
             textcoords="offset points", ha="right", va="top",
             fontsize=7.5, color="0.35")
axL.set_xlim(hist.index[0], hist.index[-1])
axL.set_ylim(0, np.nanmax(hist.values) * 1.2)
axL.xaxis.set_major_locator(mdates.MonthLocator())
axL.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axL.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
axL.grid(axis="y", color="0.85", lw=0.5)
axL.set_ylabel("Fault onsets per day")
axL.set_title("Context: last 120 days", loc="left", color="0.3")
axL.legend(frameon=False, loc="upper left", handlelength=1.8, borderaxespad=0.2)

# ---- right: 30-day forecast window ---------------------------------
axR.plot(test.index, test.values, color="0.25", lw=1.2, label="Observed", zorder=6)
for k, v in out.items():
    st = STYLE.get(k, dict(c="k", ls="-", lw=1.2, z=4))
    axR.plot(test.index, v, color=st["c"], ls=st["ls"], lw=st["lw"], zorder=st["z"],
             label=f"{k}  (MASE {res.loc[k, 'MASE']:.2f})")
for k, c in ci.items():
    lo, hi_ = np.clip(c[:, 0], 0, None), c[:, 1]
    axR.fill_between(test.index, lo, hi_, color=STYLE[k]["c"], alpha=0.05, lw=0, zorder=1)
    for edge in (lo, hi_):
        axR.plot(test.index, edge, color=STYLE[k]["c"], lw=0.7, ls=(0, (4, 3)),
                 alpha=0.55, zorder=2)

topR = max(np.nanmax(test.values), max(np.nanmax(v) for v in out.values())) * 1.35
axR.set_xlim(test.index[0], test.index[-1])
axR.set_ylim(0, topR)
axR.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
axR.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
axR.grid(axis="y", color="0.85", lw=0.5)
axR.set_title("Forecast horizon: 30 days", loc="left", color="0.3")
axR.legend(ncol=1, frameon=False, loc="upper left", handlelength=1.8,
           borderaxespad=0.2, labelspacing=0.3)

plt.savefig(f"{CFG['outdir']}/forecast.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{CFG['outdir']}/forecast.pdf", bbox_inches="tight")
plt.show()

## Daily count distribution (Figure 8)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import nbinom, poisson

y = train.dropna().values.astype(int)
mu_, var = y.mean(), y.var()
p = mu_ / var if var > mu_ else 0.5
rr = mu_ * p / (1 - p)
hi = int(np.percentile(y, 98))
above = int((y > hi).sum())
k = np.arange(0, hi + 1)

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.hist(np.clip(y, 0, hi), bins=np.arange(-0.5, hi + 1.5), density=True, color="0.86", edgecolor="white", lw=0.3, label="Observed", zorder=2)
ax.plot(k, poisson.pmf(k, mu_), color="#7f8c8d", ls="--", lw=1.3, label=f"Poisson  (mean {mu_:.1f})", zorder=3)
ax.plot(k, nbinom.pmf(k, rr, p), color="#1f6fb4", lw=1.6, label="Negative binomial", zorder=4)
ax.set_xlim(-0.5, hi)
ax.set_ylim(0, None)
ax.set_xlabel("Fault onsets per day")
ax.set_ylabel("Density")
ax.grid(axis="y", color="0.85", lw=0.5)
ax.legend(frameon=False, loc="upper right")
note = f"variance / mean = {var/mu_:.1f}\n(Poisson would be 1.0)"
if above:
    note += f"\n{above} day(s) above {hi} pooled in the last bin"
ax.annotate(note, xy=(0.97, 0.58), xycoords="axes fraction", ha="right", fontsize=8, color="0.3")
plt.tight_layout()
plt.savefig(f"{CFG['outdir']}/count_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"variance/mean = {var/mu_:.1f}  (Poisson would be 1.0);"
      f" x-axis capped at {hi}, {above} day(s) pooled")

___
**END**